In [0]:
# COMMAND -----------
# Notebook: 02_silver_transformations
# Description: Clean, parse, and enrich Bronze data into the Silver layer.

import pyspark.sql.functions as F
from pyspark.sql.types import TimestampType, LongType, DoubleType, BooleanType

# COMMAND -----------
# Step 1: Read raw data from the Bronze table
df_bronze = spark.table("ecommerce_bronze.online_retail")

# COMMAND -----------
# Step 2: Apply Silver transformations
df_silver = (
    df_bronze
    # Remove exact duplicate rows
    .dropDuplicates()
    
    # Cast InvoiceDate from string ("12/1/2010 8:26" or "yyyy-MM-dd HH:mm:ss") to Timestamp
    # Try parsing common Kaggle date format M/d/yyyy H:m
    .withColumn(
        "InvoiceTimestamp",
        F.coalesce(
            F.to_timestamp("InvoiceDate", "M/d/yyyy H:m"),
            F.to_timestamp("InvoiceDate", "yyyy-MM-dd HH:mm:ss"),
            F.to_timestamp("InvoiceDate")
        )
    )
    .withColumn("InvoiceDate", F.to_date("InvoiceTimestamp"))
    
    # Clean CustomerID: Convert Double/Float to Integer/Long or Null
    .withColumn("CustomerID", F.col("CustomerID").cast(LongType()))
    
    # Ensure numeric columns have correct data types
    .withColumn("Quantity", F.col("Quantity").cast("integer"))
    .withColumn("UnitPrice", F.col("UnitPrice").cast(DoubleType()))
    
    # Trim string descriptions and uppercase Country
    .withColumn("Description", F.trim(F.col("Description")))
    .withColumn("Country", F.trim(F.col("Country")))
    
    # Feature Engineering: Identify cancellations (InvoiceNo starting with 'C')
    .withColumn("IsCancelled", F.col("InvoiceNo").startswith("C"))
    
    # Feature Engineering: Calculate total sale value per row
    .withColumn("TotalAmount", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
    
    # Audit tracking: Add Silver processing timestamp
    .withColumn("_transformed_at", F.current_timestamp())
)

# COMMAND -----------
# Step 3: Apply quality filters
# Filter out null descriptions, non-positive unit prices, and zero quantity test rows
df_silver_clean = (
    df_silver
    .filter(F.col("Description").isNotNull() & (F.col("Description") != ""))
    .filter(F.col("UnitPrice") > 0)
)

# COMMAND -----------
# Step 4: Write cleaned data to Managed Silver Delta Table
(
    df_silver_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_silver.online_retail")
)

print("Silver table successfully created: ecommerce_silver.online_retail")

# Display a sample of cleaned records
display(spark.table("ecommerce_silver.online_retail").limit(10))

Silver table successfully created: ecommerce_silver.online_retail


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceTimestamp,IsCancelled,TotalAmount,_transformed_at
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01,2.55,17850,United Kingdom,2010-12-01T08:26:00.000Z,false,15.3,2026-08-10T12:30:27.513Z
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01,3.39,17850,United Kingdom,2010-12-01T08:26:00.000Z,false,20.34,2026-08-10T12:30:27.513Z
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01,7.65,17850,United Kingdom,2010-12-01T08:26:00.000Z,false,15.3,2026-08-10T12:30:27.513Z
536368,22913,RED COAT RACK PARIS FASHION,3,2010-12-01,4.95,13047,United Kingdom,2010-12-01T08:34:00.000Z,false,14.85,2026-08-10T12:30:27.513Z
536370,21791,VINTAGE HEADS AND TAILS CARD GAME,24,2010-12-01,1.25,12583,France,2010-12-01T08:45:00.000Z,false,30.0,2026-08-10T12:30:27.513Z
536370,22900,SET 2 TEA TOWELS I LOVE LONDON,24,2010-12-01,2.95,12583,France,2010-12-01T08:45:00.000Z,false,70.8,2026-08-10T12:30:27.513Z
536373,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01,7.65,17850,United Kingdom,2010-12-01T09:02:00.000Z,false,15.3,2026-08-10T12:30:27.513Z
536375,21068,VINTAGE BILLBOARD LOVE/HATE MUG,6,2010-12-01,1.06,17850,United Kingdom,2010-12-01T09:32:00.000Z,false,6.36,2026-08-10T12:30:27.513Z
536378,21931,JUMBO STORAGE BAG SUKI,10,2010-12-01,1.95,14688,United Kingdom,2010-12-01T09:37:00.000Z,false,19.5,2026-08-10T12:30:27.513Z
536381,22438,BALLOON ART MAKE YOUR OWN FLOWERS,1,2010-12-01,1.95,15311,United Kingdom,2010-12-01T09:41:00.000Z,false,1.95,2026-08-10T12:30:27.513Z
